# Weather Model VIS/CIG Comparison

Compares visibility and ceiling forecasts across multiple NWP models.

v1: **GFS MOS** (text bulletins) + **HRRR** (GRIB2, byte-range subset).
Architecture supports dropping in NBM, RRFS, NAM, etc. without touching
this notebook.

## 1. Environment setup

Colab doesn't have the GRIB stack pre-installed. eccodes is a C library,
and cfgrib/xarray are the Python side.

In [ ]:
!apt-get install -y libeccodes0 > /dev/null 2>&1
!pip install -q cfgrib xarray eccodes requests pandas matplotlib

## 2. Pull the project modules

Two options:
* clone from GitHub once you've pushed there
* `%%writefile` cells below (uncomment if you want a self-contained notebook)

Recommended: keep the modules in a repo so the notebook stays thin.

In [ ]:
# Option A: clone from your repo
# !git clone https://github.com/YOURUSER/wx_compare.git
# %cd wx_compare

# Option B: if the modules are already in /content (e.g. uploaded), just:
import sys
sys.path.insert(0, '/content/wx_compare')

## 3. Persistent cache via Google Drive

Mount Drive so re-runs don't re-download HRRR subsets (each is ~hundreds of KB
but adds up over many forecast hours and stations).

In [ ]:
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_ROOT = Path('/content/drive/MyDrive/wx_compare_cache')
except ImportError:
    # Not in Colab — use a local dir.
    CACHE_ROOT = Path.home() / 'wx_compare_cache'
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
print('Cache:', CACHE_ROOT)

## 4. Run the comparison

First target: **KJFK**, today's 12Z cycle, first 6 forecast hours.

In [ ]:
from datetime import datetime, timezone, timedelta
from models import GfsMos, Hrrr, Station
from compare import run_comparison, plot_comparison

# Use yesterday's 12Z so the data is guaranteed to be on NOMADS.
CYCLE = (datetime.now(timezone.utc) - timedelta(days=1)).replace(
    hour=12, minute=0, second=0, microsecond=0)

STATIONS_META = [
    Station(icao='KJFK', lat=40.6398, lon=-73.7789, elev_ft=13.0),
]
STATION_IDS = [s.icao for s in STATIONS_META]

sources = [
    GfsMos(cache_dir=CACHE_ROOT / 'gfs_mos'),
    Hrrr(cache_dir=CACHE_ROOT / 'hrrr', stations=STATIONS_META, fhours=range(0, 7)),
]

df = run_comparison(sources, CYCLE, STATION_IDS)
df.head(20)

In [ ]:
# Pivot view: side-by-side visibility per model.
df.pivot_table(index='valid_time', columns='model', values='vsby_sm', aggfunc='first')

In [ ]:
fig = plot_comparison(df, 'KJFK')
fig

## 5. Extending

To add another model — say NBM — create `models/nbm.py` subclassing
`ModelSource`, register it in `models/__init__.py`, and add it to the
`sources` list above. Nothing else changes.